# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [4]:
import asyncio
import json
import os
import time
from pathlib import Path

import pandas as pd
from openai import AsyncOpenAI

# Set your API key directly (paste your Vocareum key here)
API_KEY = "voc-1852052521403752119416a80971aa1a788.35414455"  # Replace with your actual Vocareum API key
os.environ['OPENAI_API_KEY'] = API_KEY

client = AsyncOpenAI(
    api_key=os.environ.get('OPENAI_API_KEY'),
    base_url="https://openai.vocareum.com/v1"
)

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [5]:
DATA_DIR = Path('../data')   # adjust if your folder layout differs

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])

Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [10]:
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""
    # TODO: return a messages list like [{'role': 'user', 'content': '...'}]
    prompt = f"""Extract following information from this job posting:
    1. company - the name of the hiring company
    2. role - the job title or position name
    3. years_experience_required - minimum years of experience required (or null if not stated)

    Return ONLY a JSON object with these exact field names. Do not include any extra text.
    If a field is not stated in the snippet, return null for that field.

    Job Posting:
    {snippet_text}

    JSON:"""
    
    return [{'role': 'user', 'content': prompt}]


def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include 2-3 worked examples in the prompt."""
    prompt = f"""Extract the following information from job postings:
    1. company: the name of the hiring company
    2. role: the job title or position name
    3. years_experience_required: minimum years of experience required (or null if not stated)
    
    Return ONLY a JSON object with these exact field names. If a field is not stated, return null.
    
    Here are 3 examples:
    
    EXAMPLE 1:
    Snippet: "Acme Corp is hiring a Senior Software Engineer. We need someone with 5+ years of Python experience."
    Output: {{"company": "Acme Corp", "role": "Senior Software Engineer", "years_experience_required": 5}}
    
    EXAMPLE 2:
    Snippet: "DataFlow Inc seeks a Data Analyst with 2 years minimum SQL experience. Great benefits!"
    Output: {{"company": "DataFlow Inc", "role": "Data Analyst", "years_experience_required": 2}}
    
    EXAMPLE 3:
    Snippet: "TechHub is looking for a DevOps Engineer. We welcome candidates at any experience level. No minimum years required."
    Output: {{"company": "TechHub", "role": "DevOps Engineer", "years_experience_required": null}}
    
    Now extract from this job posting:
    {snippet_text}
    
    JSON:"""
    
    return [{'role': 'user', 'content': prompt}]


def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. Use a system prompt with a persona and explicit JSON schema."""
    system_prompt = """You are an expert HR recruiter tasked with extracting structured information from job postings.

    Your job is to extract exactly three fields from each job posting:
    1. company (string): The exact name of the company hiring. Do not infer or modify.
    2. role (string): The exact job title or position name. Do not infer or modify.
    3. years_experience_required (integer or null): The minimum years of experience required.
       - If the posting states a specific number (e.g., "5 years", "3+"), extract that number as an integer.
       - If the posting says "5+ years", extract 5.
       - If the posting says "around 5 years", extract 5.
       - If the posting does NOT state a minimum years requirement, return null.
       - DO NOT infer, guess, or hallucinate a number if it is not explicitly stated.
       - DO NOT return null unless the posting clearly states no requirement.
    
    You MUST output ONLY a valid JSON object with these exact field names:
    {
      "company": <string or null>,
      "role": <string or null>,
      "years_experience_required": <integer or null>
    }
    
    Do not include any other text, explanation, or markdown formatting."""
    
    user_prompt = f"""Extract information from this job posting:
    
    {snippet_text}"""
    
    return [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt}
        ]


def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""
    prompt = f"""Extract company, role, and years_experience_required from this job posting.

    IMPORTANT: Think step by step before answering.
    1. First, identify the company name mentioned in the posting.
    2. Then, identify the job title or role.
    3. Finally, look for any mention of years of experience required:
       - If stated explicitly (e.g., "5 years", "3+"), note that number.
       - If NOT stated, note that it's missing.
    4. Return null for any field that is not stated — do not infer or guess.
    
    After thinking through these steps, output ONLY a JSON object in this format:
    {{"company": <string or null>, "role": <string or null>, "years_experience_required": <integer or null>}}
    
    Job posting:
    {snippet_text}
    
    Let's think step by step:"""
    
    return [{'role': 'user', 'content': prompt}]


STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [ ]:
def parse_response(text: str) -> dict | None:
    """Try to parse a JSON object out of the model's response. Return None if it doesn't parse.
    
    Hint: models sometimes wrap JSON in ```json ... ``` fences. Strip them first.
    """
    # TODO: extract + parse the JSON, return a dict or None
    raise NotImplementedError


async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet. Return a dict with all the captured fields."""
    # TODO: call the model, time the call, compute cost, parse the response
    raise NotImplementedError


async def run_all() -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""
    # TODO: build the task list, gather, return results
    raise NotImplementedError

In [ ]:
# Run it
results = await run_all()
print(f'Got {len(results)} results.')
results[0]

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [ ]:
def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Compare 3 fields. Case-insensitive, whitespace-trimmed for strings. Return 0, 1, 2, or 3."""
    # TODO: count exact matches (with normalisation)
    raise NotImplementedError


async def score_llm_judge(snippet_text: str, extracted: dict | None, gold: dict) -> int:
    """Use gpt-4o as a judge. Return integer 1-4.
    
    Rubric (suggested):
      4 — all three fields correct
      3 — two of three correct, no fabricated data
      2 — one of three correct, or fabricated a field
      1 — none correct or unparsable
    """
    # TODO: prompt the judge with both the gold and the extracted, ask for a 1-4 score
    raise NotImplementedError

In [ ]:
# Apply scoring to all 40 results
# TODO: loop through results, attach accuracy + parse_success + llm_judge_score to each row
scored = []   # list of result dicts with scoring fields added
print(f'Scored {len(scored)} results.')

## Step 5 — Build the comparison table

In [ ]:
df = pd.DataFrame(scored)

summary = df.groupby('strategy').agg({
    'accuracy': 'mean',
    'parse_success': 'mean',
    'llm_judge_score': 'mean',
    'cost_usd': 'sum',
    'latency_s': 'median',
}).round(3)

summary.columns = ['Accuracy (mean of 3)', 'Parse rate', 'Judge score', 'Total cost ($)', 'Latency p50 (s)']
summary

## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```